MCS 320 Quiz 5 Friday 25 September 2026

# Question 1

Let
$p = 7 x^8 - 4 x^7 + 5 x^6 - 7 x^5 + 4 x^4 - 6 x^3 - 4 x^2 + 3 x + 1$.

1. Construct the Horner form of $p$.

2. Make a fast callable object of the Horner form of $p$.

3. Compare the time it takes to evaluate $p$ at 1.0

   with the time to evaluate the fast callable of $p$ at 1.0.

   Which form is faster to evaluate?

## answer to question 1

In [1]:
p = 7*x^8 - 4*x^7 + 5*x^6 - 7*x^5 + 4*x^4 - 6*x^3 - 4*x^2 + 3*x + 1
p

7*x^8 - 4*x^7 + 5*x^6 - 7*x^5 + 4*x^4 - 6*x^3 - 4*x^2 + 3*x + 1

In [2]:
h = p.horner(x)
h

(((((((7*x - 4)*x + 5)*x - 7)*x + 4)*x - 6)*x - 4)*x + 3)*x + 1

In [3]:
f = fast_callable(h, vars=['x'])
f

In [4]:
timeit('p(x=1.0)')

625 loops, best of 3: 40.3 μs per loop

In [5]:
timeit('f(1.0)')

625 loops, best of 3: 6.79 μs per loop

We observe that the fast callable version of $p$ is faster to evaluate.

# Question 2

Consider the expression
$$
   q = \frac{x^3 + 3 y^2 + 2}{y^3 + 3 x^2 + 2}. 
$$
Draw the tree which represents $q$.

## answer to question 2

In [6]:
x, y = var('x, y')
q = (x^3 + 3*y^2 + 2)/(y^3 + 3*x^2 + 2)
q

(x^3 + 3*y^2 + 2)/(y^3 + 3*x^2 + 2)

In [7]:
q.operator()

<function mul_vararg at 0x7d0d39c51ea0>

In [8]:
q.operands()

[x^3 + 3*y^2 + 2, 1/(y^3 + 3*x^2 + 2)]

Observe how the division operator is stored.

In [9]:
q.operands()[1].operator()

<built-in function pow>

In [10]:
q.operands()[1].operands()

[y^3 + 3*x^2 + 2, -1]

We start by making all leaves of the tree.

In [11]:
Lx = LabelledOrderedTree([], 'x')
Ly = LabelledOrderedTree([], 'y')
L1 = LabelledOrderedTree([], '1')
L3 = LabelledOrderedTree([], '3')
L2 = LabelledOrderedTree([], '2')

In [12]:
q.operands()[0].operands()

[x^3, 3*y^2, 2]

In [13]:
q.operands()[0].operands()[0].operands()

[x, 3]

In [14]:
q.operands()[0].operands()[1].operands()

[y^2, 3]

In [15]:
Ly2 = LabelledOrderedTree([Ly, L2], '^')
L3y2 = LabelledOrderedTree([Ly2, L3], '*')
ascii_art(L3y2)

    *__
   /  /
  ^_ 3
 / /
y 2 

In [16]:
Lx3 = LabelledOrderedTree([Lx, L3], '^')
num = LabelledOrderedTree([Lx3, L3y2, L2], '+')
ascii_art(num)

    ____+______
   /      /   /
  ^_     *__ 2
 / /    /  /
x 3    ^_ 3 
      / /   
     y 2    

In [17]:
Lx2 = LabelledOrderedTree([Lx, L2], '^')
L3x2 = LabelledOrderedTree([Lx2, L3], '*')
Ly3 = LabelledOrderedTree([Ly, L3], '^')
den = LabelledOrderedTree([Ly3, L3x2, L2], '+')
ascii_art(den)

    ____+______
   /      /   /
  ^_     *__ 2
 / /    /  /
y 3    ^_ 3 
      / /   
     x 2    

Because division is stored as the denominator to the power -1, we invert the denominator.

In [18]:
Lm1 = LabelledOrderedTree([], '-1')
invden = LabelledOrderedTree([den, Lm1], '^')
ascii_art(invden)

          ___^____
         /       / 
    ____+______ -1
   /      /   /
  ^_     *__ 2 
 / /    /  /   
y 3    ^_ 3    
      / /      
     x 2       

In [19]:
tree = LabelledOrderedTree([num, invden], '*')
ascii_art(tree)

          _________*___________
         /                    /     
    ____+______           ___^____
   /      /   /          /       / 
  ^_     *__ 2      ____+______ -1
 / /    /  /       /      /   /
x 3    ^_ 3       ^_     *__ 2 
      / /        / /    /  /   
     y 2        y 3    ^_ 3    
                      / /      
                     x 2       